In [ ]:
from pathlib import Path
project = Path.cwd()
if project.name == 'notebooks':
    project = project.parent

Download 2 Datasets from the following data source:

1. credit.csv
2. movies_metadata.csv

[The Movies Dataset](https://www.kaggle.com/rounakbanik/the-movies-dataset)

In [ ]:

import numpy as np
import pandas as pd

In [ ]:


credits = pd.read_csv(project / 'credits.csv')
credits.head()

In [ ]:

meta = pd.read_csv(project / 'movies_metadata.csv')

In [ ]:
meta

In [ ]:
meta['release_date'] = pd.to_datetime(meta['release_date'], errors='coerce') #If ‘coerce’, then invalid parsing will be set as NaN, if ‘raise’, then invalid parsing will raise an exception, If ‘ignore’, then invalid parsing will return the input

In [ ]:
#format the date
meta['year'] = meta['release_date'].dt.year

In [ ]:
meta['year'].value_counts().sort_index()

In [ ]:

# Getting only 2017 movies as we already have movies up to the year 2016 in data 1 processed file.
# We don't have enough data for the movies from 2018, 2019 and 2020.
# We'll deal with it in the upcoming preprocessing files
new_meta = meta.loc[meta.year == 2017,['genres','id','title','year']]

In [ ]:

new_meta

In [ ]:
new_meta.dtypes

In [ ]:
#converting the "id" to integer
new_meta['id'] = new_meta['id'].astype(int)

In [ ]:
#add the new_meta data to the credit data. They all have "id" so we can merge on that.
data = pd.merge(new_meta, credits, on='id')

In [ ]:
pd.set_option('display.max_colwidth', 75)
data

In [ ]:
# we will convert the "genre", "cast" and "crew" column into a list. if we observe it carefully, we will realised that it's a list containing dictionary.
import ast
data['genres'] = data['genres'].map(lambda x: ast.literal_eval(x))  #the "literal_eval" helps to convert the string into a list
data['cast'] = data['cast'].map(lambda x: ast.literal_eval(x))
data['crew'] = data['crew'].map(lambda x: ast.literal_eval(x))

**Genres**

In [ ]:
#Now we will have to take out the individual genres like adventure, action, sci-fi, etc. using the following function


def make_genresList(x):
    gen = []
    st = " "
    for i in x:
        if i.get('name') == 'Science Fiction': #am only renaming the "Science Fiction" to "Sci-Fi" to make it the name shorter. Apart from that all other names remain the same.
            scifi = 'Sci-Fi'
            gen.append(scifi)
        else:
            gen.append(i.get('name'))
    if gen == []:
        return np.nan
    else:
        return (st.join(gen)) #then we will join them together and return the valuse

In [ ]:

#now let's apply the function on the genre column

data['genres_list'] = data['genres'].map(lambda x: make_genresList(x))

In [ ]:

data['genres_list']

**Actor 1**

In [ ]:

#let's do the same for cast

def get_actor1(x):
    casts = []
    for i in x:
        casts.append(i.get('name'))
    if casts == []:
        return np.nan
    else:
        return (casts[0])

In [ ]:

#let's apply it on the cast column for the first actor

data['actor_1_name'] = data['cast'].map(lambda x: get_actor1(x))

**Actor 2**

In [ ]:

def get_actor2(x):
    casts = []
    for i in x:
        casts.append(i.get('name'))
    if casts == [] or len(casts)<=1:
        return np.nan
    else:
        return (casts[1])

In [ ]:

#let's apply it on the cast column for the second actor
data['actor_2_name'] = data['cast'].map(lambda x: get_actor2(x))

In [ ]:

data['actor_2_name']

**Actor 3**

In [ ]:

def get_actor3(x):
    casts = []
    for i in x:
        casts.append(i.get('name'))
    if casts == [] or len(casts)<=2:
        return np.nan
    else:
        return (casts[2])

In [ ]:

data['actor_3_name'] = data['cast'].map(lambda x: get_actor3(x))

In [ ]:

data['actor_3_name']

**Directors**

In [ ]:
def get_directors(x):
    dt = []
    st = " "
    for i in x:
        if i.get('job') == 'Director':
            dt.append(i.get('name'))
    if dt == []:
        return np.nan
    else:
        return (st.join(dt))

In [ ]:

data['director_name'] = data['crew'].map(lambda x: get_directors(x))

In [ ]:

data['director_name']

In [ ]:
#selecting only the prepared data
movie = data.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres_list','title']]
movie

In [ ]:

movie.isna().sum()

In [ ]:
#drop missing values
movie = movie.dropna(how='any')

In [ ]:

movie.isna().sum()

In [ ]:
#renaming
movie = movie.rename(columns={'genres_list':'genres'})
movie = movie.rename(columns={'title':'movie_title'})

In [ ]:
#convert all movie titles to lower case
movie['movie_title'] = movie['movie_title'].str.lower()

In [ ]:
#we will be using this information later on in the tfidvectorizor
movie['comb'] = movie['actor_1_name'] + ' ' + movie['actor_2_name'] + ' '+ movie['actor_3_name'] + ' '+ movie['director_name'] +' ' + movie['genres']
movie

In [ ]:
old = pd.read_csv(project / 'data1.csv')
old

In [ ]:
old['comb'] = old['actor_1_name'] + ' ' + old['actor_2_name'] + ' '+ old['actor_3_name'] + ' '+ old['director_name'] +' ' + old['genres']

old

In [ ]:
#putting all datasets together

new = pd.concat([old, movie])
new

In [ ]:
#we are dropping the duplicated valuse and keeping only the last one, so that there will be only one type of a particular movie and not duplicates of same movie
new.drop_duplicates(subset ="movie_title", keep = 'last', inplace = True)
new

In [ ]:

#saving all the data up to 2017
new.to_csv(project / 'new_data.csv',index=False)